In [0]:
# Análisis de población, En esta tabla agregaremos un campo calculado: el crecimiento porcentual anual de la población.
# El primer año tendrá un valor nulo en population_growth_pct, porque no existe un año anterior dentro del período seleccionado.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Leer datos limpios
df = spark.table(
    "world_bank.costa_rica.silver_economic_indicators"
)

# Ventana ordenada por año
ventana = Window.orderBy("year")

gold_population = (
    df.filter(F.col("indicator_code") == "SP.POP.TOTL")
    .select(
        "year",
        F.col("value").alias("population")
    )
    .withColumn(
        "previous_population",
        F.lag("population").over(ventana)
    )
    .withColumn(
        "population_growth_pct",
        F.round(
            (
                (F.col("population") - F.col("previous_population"))
                / F.col("previous_population")
            ) * 100,
            2
        )
    )
)

gold_population.write.format("delta").mode("overwrite").saveAsTable(
    "world_bank.costa_rica.gold_population_analysis"
)

display(gold_population.orderBy("year"))